# Taller modelos de regresión lineal regularizados

Con el archivo de datos suministrado, va a crear un modelo **lineal** que prediga la variable **Weight**.

Información sobre este archivo, incluido el diccionario de datos, la puede obtener en [Kaggle](https://www.kaggle.com/datasets/samruddhim/olympics-althlete-events-analysis).

In [10]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path().resolve().parent / 'data' / 'raw'
file_name = 'athlete_events.csv'
file_path = DATA_DIR / file_name

df = pd.read_csv(file_path)
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


Haga un análisis exploratorio y descarte variables irrelevantes.

Elimine los registros duplicados.

Elimine datos nulos de la variable objetivo.

Impute a los datos nulos de la variable **Medal** la categoría **No medal**.

In [11]:
df = df.drop(columns=['ID', 'Name', 'Games'])
df = df.dropna(subset=['Weight'])
df['Medal'] = df['Medal'].fillna('No Medal')
df = df.drop_duplicates()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 207877 entries, 0 to 271115
Data columns (total 12 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Sex     207877 non-null  object 
 1   Age     207026 non-null  float64
 2   Height  206526 non-null  float64
 3   Weight  207877 non-null  float64
 4   Team    207877 non-null  object 
 5   NOC     207877 non-null  object 
 6   Year    207877 non-null  int64  
 7   Season  207877 non-null  object 
 8   City    207877 non-null  object 
 9   Sport   207877 non-null  object 
 10  Event   207877 non-null  object 
 11  Medal   207877 non-null  object 
dtypes: float64(3), int64(1), object(8)
memory usage: 20.6+ MB


In [12]:
df.isnull().sum().sort_values(ascending=False)

Height    1351
Age        851
Sex          0
Weight       0
Team         0
NOC          0
Year         0
Season       0
City         0
Sport        0
Event        0
Medal        0
dtype: int64

Cree la matriz de características y el vector objetivo. 

Parta los datos en conjuntos de entrenamiento y prueba en una proporción 80/20.

Especifique, usando `ColumnTransformer` y `Pipeline`, el procesamiento que le va a realizar a los características de su modelo. Utilice `TargetEncoder` para codificar las variables de alta cardinalidad, mayor a 5.

In [13]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Weight'])
y = df['Weight']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [14]:
df.describe(include='object').T

,count,unique,top,freq
Sex,207877,2,M,141183
Team,207877,680,United States,13735
NOC,207877,226,USA,14237
Season,207877,2,Summer,168402
City,207877,42,London,14060
Sport,207877,56,Athletics,32580
Event,207877,594,Ice Hockey Men's Ice Hockey,3810
Medal,207877,4,No Medal,177472


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import(
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    TargetEncoder,
    PowerTransformer
)

imputer = SimpleImputer(strategy='median')
ohe = OneHotEncoder(drop='if_binary', sparse_output=False)
ordinal = OrdinalEncoder(categories=[['No Medal', 'Bronze', 'Silver', 'Gold']])
te = TargetEncoder()
scaler = StandardScaler()
minmax = MinMaxScaler()
pw = PowerTransformer()

proc_age = Pipeline(steps=[
    ('imputer', imputer),
    ('pw', pw)
])
proc_height = Pipeline(steps=[
    ('imputer', imputer),
    ('scaler', scaler)
])

preprocessor = ColumnTransformer(transformers=[
    ('ohe', ohe, ['Sex', 'Season']),
    ('ordinal', ordinal, ['Medal']),
    ('te', te, ['Team', 'NOC', 'Season', 'Sport', 'Event']),
    ('proc_height', proc_height, ['Height']),
    ('minmax', minmax, ['Year']),
    ('proc_age', proc_age, ['Age'])
], remainder='drop')



Entrene un modelo de regresión lineal OLS. Reporte el error de entrenamiento y el error de prueba. Use RMSE como métrica de evaluación.

Reporte en una tabla los coeficientes asociados a cada característica del modelo.

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)

print('Train RMSE:', root_mean_squared_error(y_train, model.predict(X_train)))
print('Test RMSE:', root_mean_squared_error(y_test, model.predict(X_test)))

Train RMSE: 6.205625843090236
Test RMSE: 6.223242205004581


In [17]:
features = model.named_steps['preprocessor'].get_feature_names_out()
coefs = model.named_steps['regressor'].coef_
coef_df = pd.DataFrame({
    'feature': features,
    'coef': coefs
}).sort_values(by='coef', ascending=False)
coef_df

,feature,coef
8,proc_height__Height,5.777163
9,minmax__Year,2.756146
7,te__Event,0.732123
10,proc_age__Age,0.646188
1,ohe__Season_Winter,0.317490
2,ordinal__Medal,0.219659
4,te__NOC,0.117201
3,te__Team,0.106401
6,te__Sport,-0.117131
5,te__Season,-1.427272


Entrene un modelo Ridge. Sintonice  $\lambda$ por validación cruzada. Pruebe al menos 50 valores de  $\lambda$. 

Reporte el error promedio de validación cruzada, el valor de $\lambda$ sintonizado, el error de entrenamiento, y el error de prueba.

Reporte en una tabla los coeficientes asociados a cada característica del modelo.

In [18]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
import numpy as np

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

param_grid = {'regressor__alpha': np.logspace(-4, 4)}

grid_search = GridSearchCV(model, param_grid, scoring='neg_root_mean_squared_error')

grid_search.fit(X_train, y_train)

print('Best RMSE:', -grid_search.best_score_)
print('Best alpha:', grid_search.best_params_['regressor__alpha'])
print('Train RMSE:', root_mean_squared_error(y_train, grid_search.best_estimator_.predict(X_train)))
print('Test RMSE:', root_mean_squared_error(y_test, grid_search.best_estimator_.predict(X_test)))

Best RMSE: 6.2331664213618385
Best alpha: 0.019306977288832496
Train RMSE: 6.205509477355103
Test RMSE: 6.2232608250977846


In [20]:
features = grid_search.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
coefs = grid_search.best_estimator_.named_steps['regressor'].coef_
coef_df = pd.DataFrame({
    'feature': features,
    'coef': coefs
}).sort_values(by='coef', key=abs, ascending=False)
coef_df

,feature,coef
8,proc_height__Height,5.775186
9,minmax__Year,2.758134
5,te__Season,-1.520689
0,ohe__Sex_M,-1.431892
7,te__Event,0.732147
10,proc_age__Age,0.647660
1,ohe__Season_Winter,0.327694
2,ordinal__Medal,0.220158
6,te__Sport,-0.117192
4,te__NOC,0.112955


Entrene un modelo Lasso. Sintonice  $\lambda$ por validación cruzada. Pruebe al menos 50 valores de  $\lambda$.

Reporte el error promedio de validación cruzada, el valor de $\lambda$ sintonizado, el error de entrenamiento, y el error de prueba.

Reporte en una tabla los coeficientes asociados a cada característica del modelo.

In [21]:
from sklearn.linear_model import Lasso

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso())
])

param_grid = {
    'regressor__alpha': np.logspace(-4, 4)
}

grid_search = GridSearchCV(
    model,
    param_grid,
    scoring='neg_root_mean_squared_error'
)

grid_search.fit(X_train, y_train)

print('Best RMSE:', -grid_search.best_score_)
print('Best alpha:', grid_search.best_params_['regressor__alpha'])
print('Train RMSE:', root_mean_squared_error(y_train, grid_search.best_estimator_.predict(X_train)))
print('Test RMSE:', root_mean_squared_error(y_test, grid_search.best_estimator_.predict(X_test)))

Best RMSE: 6.23316555841859
Best alpha: 0.00014563484775012445
Train RMSE: 6.205891737545726
Test RMSE: 6.223262855963363


In [22]:
features = grid_search.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
coefs = grid_search.best_estimator_.named_steps['regressor'].coef_
coef_df = pd.DataFrame({
    'feature': features,
    'coef': coefs
}).sort_values(by='coef', key=abs, ascending=False)
coef_df

,feature,coef
8,proc_height__Height,5.771099
9,minmax__Year,2.741213
5,te__Season,-1.540220
0,ohe__Sex_M,-1.431889
7,te__Event,0.732239
10,proc_age__Age,0.646350
1,ohe__Season_Winter,0.334506
2,ordinal__Medal,0.221383
4,te__NOC,0.129945
6,te__Sport,-0.116747


Use un modelo Lasso para hacer selección de características preservando únicamente 5 características en su modelo.

Con las características seleccionadas entrene nuevamente un modelo Ridge, sintonizando $\lambda$. 

Reporte el error promedio de validación cruzada, el valor de $\lambda$ sintonizado, el error de entrenamiento, y el error de prueba.

Reporte los coeficientes del modelo.

In [23]:
selector = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=1))
])

selector.fit(X_train, y_train)

features = selector.named_steps['preprocessor'].get_feature_names_out()
coefs = selector.named_steps['regressor'].coef_
coef_df = pd.DataFrame({
    'feature': features,
    'coef': coefs
}).sort_values(by='coef', key=abs, ascending=False)
coef_df

,feature,coef
8,proc_height__Height,3.969954
7,te__Event,0.764719
4,te__NOC,0.193166
3,te__Team,0.047330
6,te__Sport,-0.003499
0,ohe__Sex_M,-0.000000
1,ohe__Season_Winter,0.000000
2,ordinal__Medal,0.000000
5,te__Season,0.000000
9,minmax__Year,0.000000


In [24]:
selected_faetures = coef_df[coef_df['coef'] != 0]['feature']
selected_faetures

8    proc_height__Height
7              te__Event
4                te__NOC
3               te__Team
6              te__Sport
Name: feature, dtype: object

In [25]:
X_sel = X[['Height', 'Event', 'NOC', 'Team', 'Sport']]

X_train_sel, X_test_sel, y_train_sel, y_test_sel = train_test_split(X_sel, y, test_size=0.2, random_state=1)

preprocessor = ColumnTransformer(transformers=[
    ('te', te, ['Team', 'NOC', 'Event', 'Sport']),
    ('proc_height', proc_height, ['Height']),
    ], remainder='drop')

model_sel = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

param_grid = {
    'regressor__alpha': np.logspace(-4, 4)
}

grid_search_sel = GridSearchCV(
    model_sel,
    param_grid,
    scoring='neg_root_mean_squared_error'
)

grid_search_sel.fit(X_train_sel, y_train_sel)
print('Best RMSE:', -grid_search_sel.best_score_)
print('Best alpha:', grid_search_sel.best_params_['regressor__alpha'])
print('Train RMSE:', root_mean_squared_error(y_train_sel, grid_search_sel.best_estimator_.predict(X_train_sel)))
print('Test RMSE:', root_mean_squared_error(y_test_sel, grid_search_sel.best_estimator_.predict(X_test_sel)))

Best RMSE: 6.311728551809898
Best alpha: 0.8286427728546842
Train RMSE: 6.285101050943871
Test RMSE: 6.300880293144062


In [26]:
features = grid_search_sel.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
coefs = grid_search_sel.best_estimator_.named_steps['regressor'].coef_
coef_df = pd.DataFrame({
    'feature': features,
    'coef': coefs
}).sort_values(by='coef', key=abs, ascending=False)
coef_df

,feature,coef
4,proc_height__Height,5.728925
2,te__Event,0.693352
1,te__NOC,0.131671
0,te__Team,0.114695
3,te__Sport,-0.063601
